In [2]:
import json
import numpy as np
import pandas as pd
from types import SimpleNamespace
from verbatim_rag.extractors import LLMSpanExtractor, SemanticHighlightExtractor

# Dev set laden
def load_jsonl(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

dev_answerable = load_jsonl('../../data/clapnq_dev_answerable.jsonl')
print(f"Dev answerable: {len(dev_answerable)}")

# Eigene Annotationen laden
with open("../phase3/annotations_gold.json", "r", encoding="utf-8") as f:
    annotations = json.load(f)

# Nur die tatsächlich annotierten (mit Spans)
annotations_with_spans = [a for a in annotations if a["passage_spans"] or a["title_spans"]]
print(f"Annotationen gesamt: {len(annotations)}")
print(f"Annotationen mit Spans: {len(annotations_with_spans)}")

Dev answerable: 300
Annotationen gesamt: 100
Annotationen mit Spans: 87


In [3]:
# Dev set als dict für schnellen Lookup über question text
dev_by_question = {ex['input']: ex for ex in dev_answerable}

# Matchen
matched_examples = []
not_found = []

for ann in annotations_with_spans:
    question = ann["question"]
    if question in dev_by_question:
        ex = dev_by_question[question]
        matched_examples.append({
            "annotation": ann,
            "dev_example": ex,
        })
    else:
        not_found.append(ann["question_id"])

print(f"Gematcht: {len(matched_examples)}")
if not_found:
    print(f"Nicht gefunden: {not_found}")

Gematcht: 87


In [4]:
# Die gleichen Funktionen wie in Phase 2

def find_span_in_passage(span, passage):
    idx = passage.find(span)
    if idx != -1:
        return (idx, idx + len(span))
    span_stripped = span.strip()
    idx = passage.find(span_stripped)
    if idx != -1:
        return (idx, idx + len(span_stripped))
    return None


def compute_span_iou_metrics(extracted_spans, gt_spans, passage_text):
    """Combined IoU: all extracted chars vs all GT chars."""
    gt_chars = set()
    for span in gt_spans:
        pos = find_span_in_passage(span, passage_text)
        if pos is not None:
            gt_chars.update(range(pos[0], pos[1]))

    extracted_chars = set()
    for span in extracted_spans:
        pos = find_span_in_passage(span, passage_text)
        if pos is not None:
            extracted_chars.update(range(pos[0], pos[1]))

    if not gt_chars and not extracted_chars:
        return {"iou": 1.0}
    if not gt_chars or not extracted_chars:
        return {"iou": 0.0}

    intersection = len(gt_chars & extracted_chars)
    union = len(gt_chars | extracted_chars)
    return {"iou": intersection / union}


def compute_token_f1(extracted_spans, gt_spans):
    import re
    from collections import Counter
    pred_tokens = []
    for s in extracted_spans:
        pred_tokens.extend(re.findall(r'\w+', s.lower()))
    gt_tokens = []
    for s in gt_spans:
        gt_tokens.extend(re.findall(r'\w+', s.lower()))

    if not pred_tokens and not gt_tokens:
        return {"token_precision": 1.0, "token_recall": 1.0, "token_f1": 1.0}
    if not pred_tokens or not gt_tokens:
        return {"token_precision": 0.0, "token_recall": 0.0, "token_f1": 0.0}

    pred_counts = Counter(pred_tokens)
    gt_counts = Counter(gt_tokens)
    overlap = sum(min(c, gt_counts.get(t, 0)) for t, c in pred_counts.items())

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gt_tokens)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"token_precision": precision, "token_recall": recall, "token_f1": f1}


def compute_coverage_and_overextraction(extracted_spans, gt_spans, passage_text):
    gt_chars = set()
    for s in gt_spans:
        pos = find_span_in_passage(s, passage_text)
        if pos:
            gt_chars.update(range(pos[0], pos[1]))
    pred_chars = set()
    for s in extracted_spans:
        pos = find_span_in_passage(s, passage_text)
        if pos:
            pred_chars.update(range(pos[0], pos[1]))

    if not gt_chars and not pred_chars:
        return {"coverage": 1.0, "over_extraction": 0.0}
    coverage = len(gt_chars & pred_chars) / len(gt_chars) if gt_chars else 0.0
    over_ext = len(pred_chars - gt_chars) / len(pred_chars) if pred_chars else 0.0
    return {"coverage": coverage, "over_extraction": over_ext}


print("Metriken-Funktionen definiert ✓")

Metriken-Funktionen definiert ✓


In [5]:
llm_extractor = LLMSpanExtractor(span_match_mode="fuzzy")
semantic_extractor = SemanticHighlightExtractor(threshold=0.5, output_mode="sentences")
semantic_span_extractor = SemanticHighlightExtractor(threshold=0.5, output_mode="spans")

extraction_results = []

for i, m in enumerate(matched_examples):
    ex = m["dev_example"]
    ann = m["annotation"]
    passage_text = ex["passages"][0]["text"]
    question = ex["input"]

    search_results = [SimpleNamespace(text=passage_text)]

    # LLM Extractor
    try:
        llm_spans = llm_extractor.extract_spans(question, search_results)
        llm_extracted = list(llm_spans.values())[0] if llm_spans else []
    except:
        llm_extracted = []

    # Semantic Extractor
    try:
        sem_spans = semantic_extractor.extract_spans(question, search_results)
        sem_extracted = list(sem_spans.values())[0] if sem_spans else []
    except:
        sem_extracted = []

    # Semantic Extractor (span mode)
    try:
        sem_span_spans = semantic_span_extractor.extract_spans(question, search_results)
        sem_span_extracted = list(sem_span_spans.values())[0] if sem_span_spans else []
    except:
        sem_span_extracted = []

    extraction_results.append({
        "question_id": ann["question_id"],
        "question": question,
        "passage_text": passage_text,
        "llm_extracted": llm_extracted,
        "sem_extracted": sem_extracted,
        "sem_span_extracted": sem_span_extracted,
        "gt_sentences": ex["output"][0]["selected_sentences"],  # CLAPNQ GT
        "my_spans": [s["text"] for s in ann["passage_spans"]],  # Eigene Annotationen
    })

    if (i + 1) % 20 == 0:
        print(f"  {i + 1}/{len(matched_examples)} done")

print(f"\nExtraction fertig: {len(extraction_results)} Beispiele")

Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 51.09it/s]


[OpenProvenceModel] Model inference time: 2.61s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 143.33it/s]


[OpenProvenceModel] Model inference time: 0.95s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 486.69it/s]


...

[OpenProvenceModel] Model inference time: 1.11s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 284.77it/s]


[OpenProvenceModel] Model inference time: 1.12s (1 blocks)


Prepare contexts: 100%|██████████| 1/1 [00:00<00:00, 491.08it/s]


[OpenProvenceModel] Model inference time: 0.69s (1 blocks)

Extraction fertig: 87 Beispiele


In [6]:
def evaluate(results, extractor_key, gt_key, passage_key="passage_text"):
    ious, token_f1s, token_ps, token_rs, coverages, over_exts = [], [], [], [], [], []
    n_empty = 0
    span_counts = []

    for r in results:
        extracted = r[extractor_key]
        gt = r[gt_key]
        passage = r[passage_key]

        span_counts.append(len(extracted))
        if not extracted:
            n_empty += 1
            ious.append(0.0)
            token_f1s.append(0.0)
            token_ps.append(0.0)
            token_rs.append(0.0)
            coverages.append(0.0)
            over_exts.append(0.0)
            continue

        iou = compute_span_iou_metrics(extracted, gt, passage)
        ious.append(iou["iou"])

        tf = compute_token_f1(extracted, gt)
        token_f1s.append(tf["token_f1"])
        token_ps.append(tf["token_precision"])
        token_rs.append(tf["token_recall"])

        cov = compute_coverage_and_overextraction(extracted, gt, passage)
        coverages.append(cov["coverage"])
        over_exts.append(cov["over_extraction"])

    return {
        "mean_iou":        np.mean(ious),
        "iou_ge_30":       np.mean([i >= 0.3 for i in ious]),
        "iou_ge_50":       np.mean([i >= 0.5 for i in ious]),
        "iou_ge_70":       np.mean([i >= 0.7 for i in ious]),
        "iou_ge_90":       np.mean([i >= 0.9 for i in ious]),
        "iou_exact":       np.mean([i == 1.0 for i in ious]),
        "token_precision": np.mean(token_ps),
        "token_recall":    np.mean(token_rs),
        "token_f1":        np.mean(token_f1s),
        "coverage":        np.mean(coverages),
        "over_extraction": np.mean(over_exts),
        "n_empty":         n_empty,
        "avg_spans":       np.mean(span_counts),
    }

# 6 Kombinationen
llm_vs_clapnq      = evaluate(extraction_results, "llm_extracted",      "gt_sentences")
sem_vs_clapnq      = evaluate(extraction_results, "sem_extracted",      "gt_sentences")
sem_span_vs_clapnq = evaluate(extraction_results, "sem_span_extracted", "gt_sentences")
llm_vs_mine        = evaluate(extraction_results, "llm_extracted",      "my_spans")
sem_vs_mine        = evaluate(extraction_results, "sem_extracted",      "my_spans")
sem_span_vs_mine   = evaluate(extraction_results, "sem_span_extracted", "my_spans")

print("Evaluation fertig ✓")

Evaluation fertig ✓


In [7]:
from tabulate import tabulate

metrics = [
    ("Mean IoU",           "mean_iou",        False),
    ("IoU $\\geq$ 0.3",   "iou_ge_30",       True),
    ("IoU $\\geq$ 0.5",   "iou_ge_50",       True),
    ("IoU $\\geq$ 0.7",   "iou_ge_70",       True),
    ("IoU $\\geq$ 0.9",   "iou_ge_90",       True),
    ("IoU $=$ 1.0 (exact)","iou_exact",       True),
    ("Token Precision",    "token_precision",  False),
    ("Token Recall",       "token_recall",     False),
    ("Token F1",           "token_f1",         False),
    ("Coverage",           "coverage",         True),
    ("Over-Extraction",    "over_extraction",  True),
    ("Empty extractions",  "n_empty",          False),
    ("Avg spans/example",  "avg_spans",        False),
]

cols = [
    ("LLM / CLAPNQ",   llm_vs_clapnq),
    ("SEM / CLAPNQ",   sem_vs_clapnq),
    ("SEM-S / CLAPNQ", sem_span_vs_clapnq),
    ("LLM / Mine",     llm_vs_mine),
    ("SEM / Mine",     sem_vs_mine),
    ("SEM-S / Mine",   sem_span_vs_mine),
]

def fmt(v, pct):
    if isinstance(v, int):
        return str(v)
    return f"{v*100:.1f}\\%" if pct else f"{v:.3f}"

rows = []
for label, key, pct in metrics:
    row = [label] + [fmt(c[key], pct) for _, c in cols]
    rows.append(row)

headers = ["Metric"] + [c[0] for c in cols]

print(tabulate(rows, headers=headers, tablefmt="github"))
print("\n--- LaTeX ---")
print(tabulate(rows, headers=headers, tablefmt="latex_booktabs"))

| Metric              | LLM / CLAPNQ   | SEM / CLAPNQ   | SEM-S / CLAPNQ   | LLM / Mine   | SEM / Mine   | SEM-S / Mine   |
|---------------------|----------------|----------------|------------------|--------------|--------------|----------------|
| Mean IoU            | 0.542          | 0.420          | 0.426            | 0.516        | 0.390        | 0.402          |
| IoU $\geq$ 0.3      | 85.1\%         | 64.4\%         | 65.5\%           | 78.2\%       | 62.1\%       | 63.2\%         |
| IoU $\geq$ 0.5      | 54.0\%         | 41.4\%         | 40.2\%           | 47.1\%       | 35.6\%       | 36.8\%         |
| IoU $\geq$ 0.7      | 27.6\%         | 24.1\%         | 23.0\%           | 32.2\%       | 19.5\%       | 18.4\%         |
| IoU $\geq$ 0.9      | 8.0\%          | 10.3\%         | 11.5\%           | 11.5\%       | 6.9\%        | 6.9\%          |
| IoU $=$ 1.0 (exact) | 1.1\%          | 10.3\%         | 0.0\%            | 0.0\%        | 0.0\%        | 4.6\%          |
| Token 

In [8]:
# ============================================================
# Annotation Comparison: CLAPNQ sentence-level vs. Manual spans
# ============================================================

import numpy as np

char_counts_clapnq = []
char_counts_manual = []
sent_counts_clapnq = []
span_counts_manual = []
overlap_ratios     = []   # wie viel CLAPNQ-Text in deinen Spans enthalten ist

for r in extraction_results:
    passage = r["passage_text"]

    # CLAPNQ GT
    clapnq_sents = r["gt_sentences"]
    clapnq_text  = " ".join(clapnq_sents)
    clapnq_chars = set()
    for s in clapnq_sents:
        pos = find_span_in_passage(s, passage)
        if pos:
            clapnq_chars.update(range(pos[0], pos[1]))

    # Manual spans
    manual_spans = r["my_spans"]
    manual_text  = " ".join(manual_spans)
    manual_chars = set()
    for s in manual_spans:
        pos = find_span_in_passage(s, passage)
        if pos:
            manual_chars.update(range(pos[0], pos[1]))

    char_counts_clapnq.append(len(clapnq_chars))
    char_counts_manual.append(len(manual_chars))
    sent_counts_clapnq.append(len(clapnq_sents))
    span_counts_manual.append(len(manual_spans))

    # Overlap: wie viel von CLAPNQ steckt in Manual?
    if clapnq_chars and manual_chars:
        overlap = len(clapnq_chars & manual_chars) / len(clapnq_chars)
    else:
        overlap = 0.0
    overlap_ratios.append(overlap)

# Ausgabe
stats = [
    ["Avg GT sentences (CLAPNQ)",  f"{np.mean(sent_counts_clapnq):.2f}"],
    ["Avg GT spans (Manual)",       f"{np.mean(span_counts_manual):.2f}"],
    ["Avg GT chars (CLAPNQ)",       f"{np.mean(char_counts_clapnq):.1f}"],
    ["Avg GT chars (Manual)",       f"{np.mean(char_counts_manual):.1f}"],
    ["Avg char reduction",          f"{np.mean([c-m for c,m in zip(char_counts_clapnq, char_counts_manual)]):.1f}"],
    ["Avg overlap (CLAPNQ in Manual)", f"{np.mean(overlap_ratios)*100:.1f}%"],
]

print(tabulate(stats, headers=["Statistic", "Value"], tablefmt="github"))
print("\n--- LaTeX ---")
print(tabulate(stats, headers=["Statistic", "Value"], tablefmt="latex_booktabs"))

| Statistic                      | Value   |
|--------------------------------|---------|
| Avg GT sentences (CLAPNQ)      | 2.84    |
| Avg GT spans (Manual)          | 2.20    |
| Avg GT chars (CLAPNQ)          | 426.0   |
| Avg GT chars (Manual)          | 297.4   |
| Avg char reduction             | 128.6   |
| Avg overlap (CLAPNQ in Manual) | 55.1%   |

--- LaTeX ---
\begin{tabular}{ll}
\toprule
 Statistic                      & Value   \\
\midrule
 Avg GT sentences (CLAPNQ)      & 2.84    \\
 Avg GT spans (Manual)          & 2.20    \\
 Avg GT chars (CLAPNQ)          & 426.0   \\
 Avg GT chars (Manual)          & 297.4   \\
 Avg char reduction             & 128.6   \\
 Avg overlap (CLAPNQ in Manual) & 55.1\%   \\
\bottomrule
\end{tabular}


In [9]:
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Annotation Comparison"

# Header
headers = [
    "Question",
    "CLAPNQ GT (sentences)",
    "Manual GT (spans)",
    "CLAPNQ chars",
    "Manual chars",
    "Char reduction",
    "Overlap %",
    "Category",      # manuell ausfüllen
    "Comment",       # manuell ausfüllen
]
ws.append(headers)

# Header-Styling
header_fill = PatternFill("solid", fgColor="2F5496")
header_font = Font(color="FFFFFF", bold=True)
for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(wrap_text=True, vertical="top")

# Kategorien als Dropdown-Kommentar (zur manuellen Auswahl)
# "CLAPNQ too coarse" / "CLAPNQ missing span" / "Manual too narrow"
# / "Both equivalent" / "CLAPNQ has extra irrelevant content"

# Daten
for r in extraction_results:
    passage       = r["passage_text"]
    clapnq_sents  = r["gt_sentences"]
    manual_spans  = r["my_spans"]

    clapnq_chars = set()
    for s in clapnq_sents:
        pos = find_span_in_passage(s, passage)
        if pos:
            clapnq_chars.update(range(pos[0], pos[1]))

    manual_chars = set()
    for s in manual_spans:
        pos = find_span_in_passage(s, passage)
        if pos:
            manual_chars.update(range(pos[0], pos[1]))

    overlap = (len(clapnq_chars & manual_chars) / len(clapnq_chars) * 100
               if clapnq_chars else 0.0)
    char_reduction = len(clapnq_chars) - len(manual_chars)

    ws.append([
        r["question"],
        "\n".join(clapnq_sents),
        "\n".join(manual_spans),
        len(clapnq_chars),
        len(manual_chars),
        char_reduction,
        f"{overlap:.1f}%",
        "",   # Category – manuell ausfüllen
        "",   # Comment  – manuell ausfüllen
    ])

# Spaltenbreiten
col_widths = [40, 60, 60, 12, 12, 14, 10, 25, 40]
for i, width in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = width

# Zeilenhöhe und Wrap
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical="top")

wb.save("annotation_comparison.xlsx")
print("Gespeichert: annotation_comparison.xlsx")

Gespeichert: annotation_comparison.xlsx


In [10]:
# ============================================================================
# CELL 7: Extraction Results speichern für Phase 5
# ============================================================================

output = []
for r in extraction_results:
    output.append({
        "question_id":      r["question_id"],
        "question":         r["question"],
        "passage_text":     r["passage_text"],
        "llm_extracted":    r["llm_extracted"],
        "sem_extracted":    r["sem_extracted"],
        "sem_span_extracted": r["sem_span_extracted"],
        "gt_sentences":     r["gt_sentences"],
        "my_spans":         r["my_spans"],
    })

with open("phase4_extraction_results.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Gespeichert: phase4_extraction_results.json ({len(output)} Einträge)")

Gespeichert: phase4_extraction_results.json (87 Einträge)
